In [2]:
%env HF_HUB_DISABLE_XET=1
from workshop.llm import (
    Gedaechtnis, 
    Sprachmodell, 
)

SMOL_LM2_BASE = {
    "s": "HuggingFaceTB/SmolLM2-135M",
    "m": "HuggingFaceTB/SmolLM2-360M",
    "l": "HuggingFaceTB/SmolLM2-1.7B"
}

SMOL_LM2_INSTRUCT = {
    "s": "HuggingFaceTB/SmolLM2-135M-Instruct",
    "m": "HuggingFaceTB/SmolLM2-360M-Instruct",
    "l": "HuggingFaceTB/SmolLM2-1.7B-Instruct"
}

a = SMOL_LM2_BASE["l"]
b = SMOL_LM2_INSTRUCT["l"]

env: HF_HUB_DISABLE_XET=1


## Nicht alle Modelle sind gleich...

### Unser erstes Sprachmodell

In [7]:
modell_1 = Sprachmodell(a, device="cpu")

config.json:   0%|          | 0.00/635 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.66k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [ ]:
modell_1.prompt(prompt="Isaac Newton discovered that gravity")

In [ ]:
modell_1.prompt(prompt="What is the Pythagoeran theorem?")

### Ein anderes Modell

In [3]:
modell_2 = Sprachmodell(b, device="cpu")

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

In [ ]:
modell_2.prompt(prompt="Isaac Newton discovered that gravity")

In [ ]:
modell_2.prompt(prompt="What is the Pythagoeran theorem?")

#### Und auf Deutsch?

In [ ]:
modell_2.prompt(prompt="Was ist der Satz des Pythagoras?", max_tokens=256)

## In-Kontext Lernen | Few Shot Prompting

In [5]:
#basis_modell = modell_1
chat_modell = modell_2

Aufgabe: Übersetzen vom Englischen ins Deutsche mit einem Base Modell. Idee dahinter: Modell benötigt Struktur -> Few Shot Prompting

In [ ]:
prompt = """
Hier schreiben Sie Ihren Prompt, der das BASIS Modell dazu bringt, 
Fachbegriffe aus der Physik vom Englischen ins Deutsche zu übersetzen.
"""

basis_modell.prompt(prompt=prompt, max_tokens=3)

## Die Chat-Illusion & Instruktions-Tuning

In [ ]:
chat_modell.prompt(prompt="Explain the Pythagorean theorem in one sentence.")

In [ ]:
chat_modell.prompt(prompt="Now give me only the formula.")

*surprised pikachu face* -> Modell hat keine Ahnung welche Formel, wieso?

In [ ]:
chat_modell.prompt_auto(prompt="Explain the Pythagorean theorem in one sentence.")

In [5]:
chat_modell.prompt_auto(prompt="Now give me only the formula.")

Sure, the formula for the area of a circle is:

A = πr²

Where:
- A is the area of the circle
- π (pi) is a mathematical constant approximately equal to 3.14159
- r is the radius of the circle

### Wie "erinnern" sich Sprachmodelle?

In [6]:
chat_modell.gedaechtnis.zeige()

╭─ User ──────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Now give me only the formula.                                                                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ Assistant ─────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Sure, the formula for the area of a circle is:                                                                  │
│                                                                                                                 │
│ A = πr²                                                                                                         │
│                                                                                                                 │
│ Where:                                                                                                          │
│                                                                                                                 │
│  • A is the area of the circle                                                                                  │
│  • π (pi) is a mathematical constant approximately equal to 3.14159                                             │
│  • r is the radius of the circle                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Was sieht das Modell tatsächlich? -> Zeige Chat-Template. Hier drauf zu achten: 1. System Prompt ist vorhanden! 2. Template endet mit <|im_start|>assistant ---> Modell produziert solange Token bis <|im_end|> generiert wird!

In [ ]:
print(chat_modell.zeige_prompt(chat_modell.gedaechtnis))

<|im_start|>system
You are a space cowboy<|im_end|>
<|im_start|>user
What did you do today?<|im_end|>
<|im_start|>assistant
Well, partner, I rode out to the ranch this mornin', checkin' on the herd and makin' sure they're all healthy and happy. After that, I spent some time fixin' fences and makin' sure they're all secure. Then I headed back to the house for a bite to eat and a cold drink. After that, I spent some time fixin' my boots and makin' sure they're in good shape for the next day's work. Now, I'm just sittin' here, enjoyin' the peace and quiet of the ranch.<|im_end|>
<|im_start|>assistant



##  System Prompts

Was ist der "Sinn" hinter System-Prompts? --> Verhaltenssteuerung des Modells

In [ ]:
chat_modell.gedaechtnis.loesche()
chat_modell.prompt_auto(prompt="What did you do today?")
print(chat_modell.zeige_prompt(chat_modell.gedaechtnis))

In [ ]:
chat_modell.gedaechtnis.loesche()
chat_modell.gedaechtnis.neuer_eintrag(rolle="system", inhalt="You are a cowboy.")
chat_modell.prompt_auto(prompt="What did you do today?")
print(chat_modell.zeige_prompt(chat_modell.gedaechtnis))

## Decoding Parameter

Generieren Sprachmodelle immer dasselbe?
In Folien sollte erklärt werden wie Sampling funktioniert.
Inhalte: Determinismus in LLMs, Temperatur, Nukleus-Sampling

In [ ]:
prompt = "Write a creative metaphor for a quadratic equation."
max_tokens = 64

### Temperatur = 0.0

In [ ]:
for i in range(3):
    chat_modell.prompt(prompt=prompt, max_tokens=max_tokens, temperature=0.0)

### Temperatur = 0.7

In [ ]:
for i in range(3):
    chat_modell.prompt(prompt=prompt, max_tokens=max_tokens, temperature=0.7)

### Temperatur = 1.8

In [ ]:
for i in range(3):
    chat_modell.prompt(prompt=prompt, max_tokens=max_tokens, temperature=1.8)

### Temperatur & Nukleus-Sampling Visualisiert

Eine kleine interaktive Visualisierung

In [ ]:
from workshop.viz import visualisiere_sampling
visualisiere_sampling(modell=chat_modell)

## Attention is all you need!

Ein Sprachmodell zerlegt einen Text in Tokens und wandelt jedes Token in einen Vektor um, also eine lange Liste von Zahlen, die seine Bedeutung repräsentiert. Das Problem: Ein einzelnes Token kennt seinen Kontext nicht. „Bank" kann ein Sitzmöbel oder ein Geldinstitut sein, und erst die umgebenden Wörter entscheiden das. **Attention** ist der Mechanismus, mit dem jedes Token Informationen von anderen Tokens im Text einsammelt. Eingeführt wurde er 2017 im Artikel *„Attention Is All You Need"*, der Grundlage aller heutigen Sprachmodelle.

**Die Idee: Suchanfrage, Schlagwort, Inhalt.** Aus jedem Token-Vektor berechnet das Modell (durch Multiplikation mit gelernten Matrizen $W_Q, W_K, W_V$) drei neue Vektoren:

- **Query** ($q$): Wonach sucht dieses Token?
- **Key** ($k$): Was hat dieses Token zu bieten?
- **Value** ($v$): Welche Information gibt es weiter, wenn es ausgewählt wird?

Wie gut eine Query zu einem Key passt, misst das Skalarprodukt $q \cdot k$. Für alle Tokens gleichzeitig geschrieben ergibt sich die zentrale Formel:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

Die Division durch $\sqrt{d_k}$ (die Länge der Key-Vektoren) hält die Zahlen in einem handlichen Bereich. Die Softmax-Funktion macht aus den Passungswerten **Gewichte zwischen 0 und 1, die sich zu 1 addieren**, ähnlich wie Prozentanteile. Jedes Token erhält am Ende einen gewichteten Mittelwert der Values aller Tokens, auf die es achtet. Genau diese Gewichte werden wir gleich sichtbar machen.

<div class="alert alert-secondary">
<details>
<summary>Kausale Attention.</summary>

Ein Sprachmodell lernt, das jeweils nächste Token vorherzusagen. Dürfte es dabei in die Zukunft schauen, wäre das gemogelt. Deshalb darf jedes Token nur auf sich selbst und auf frühere Tokens achten. Technisch werden die Werte für spätere Tokens vor der Softmax auf $-\infty$ gesetzt und erhalten so das Gewicht 0.
</div>

<div class="alert alert-secondary">
<details>
<summary>Multi-Head Attention.</summary>

Ein einzelner Attention-Mechanismus könnte nur eine Art von Beziehung abbilden. Deshalb arbeiten in jeder Schicht mehrere **Heads** parallel, jeder mit eigenen Matrizen $W_Q, W_K, W_V$. So kann sich ein Head z.B. auf Grammatik spezialisieren, ein anderer auf Wiederholungen. Ihre Ergebnisse werden zusammengeführt und an die nächste Schicht weitergegeben.
</div>

In [6]:
prompt = "The capital of France is Paris."
chat_modell.zeige_attention(prompt, schicht = [-1])

**Schicht 23** (32 Heads)

**So lesen Sie die Visualisierung:**

- Oben sehen Sie eine kleine Kachel pro Head. Fahren Sie mit der Maus darüber oder klicken Sie, um den Head groß anzuzeigen.
- **Zeilen** sind die Tokens, die „hinschauen" (Query), **Spalten** die Tokens, auf die geschaut wird (Key). Je kräftiger die Farbe, desto höher das Gewicht.
- Das graue Dreieck oben rechts ist die Zukunft, die durch Causal Attention verdeckt ist.
- Viele Heads legen den Großteil ihrer Aufmerksamkeit auf das **erste Token**. Das ist eine Art Parkplatz: Hat ein Head an einer Stelle nichts zu tun, muss er seine Aufmerksamkeit trotzdem irgendwo hinlegen (die Gewichte müssen sich zu 1 addieren). Das erste Token ist von überall aus sichtbar, und das Modell lernt, dass Hinschauen dort „nichts bewirkt". Interessant sind daher vor allem die Heads, die *woanders* hinschauen.

### Typische Muster: Previous Token Heads

Ein **Previous Token Head** macht etwas sehr Einfaches: Jedes Token schaut fast ausschließlich auf das Token direkt davor. In der Visualisierung erkennt man das an einer kräftigen Linie **direkt unterhalb der Diagonalen**.

<div class="alert alert-secondary">
<details>
<summary>Warum ist das nützlich?</summary>

Nach diesem Head „weiß" jedes Token, welches Token ihm vorausging. Das ist wichtiger, als es klingt:

- **Wörter bestehen oft aus mehreren Tokens.** Ein seltener Name wie „Thornbury" wird z.B. in „Thorn" und „bury" zerlegt. Erst wenn „bury" weiß, dass „Thorn" davorstand, kann das Modell das ganze Wort verarbeiten.
- **Bedeutung hängt vom Nachbarn ab.** „York" nach „New" meint etwas anderes als „York" allein.
- **Er ist ein Baustein für komplexere Muster.** Heads späterer Schichten können auf die Information zurückgreifen, die der Previous Token Head eingesammelt hat. Das wichtigste Beispiel dafür sehen wir im nächsten Abschnitt.
</div>

In [ ]:
prompt = "todo"
chat_modell.zeige_attention(prompt, schicht = [-1])

### Typische Muster: Induction Heads

Ein **Induction Head** setzt eine Kopierregel um: *„Ich bin bei Token A. Wo kam A schon einmal vor? Welches Token folgte damals darauf? Dann sage ich das jetzt wieder voraus."* Kurz: Aus dem Muster **[A][B] … [A]** folgt die Vorhersage **[B]**.

Dafür arbeiten zwei Heads in verschiedenen Schichten zusammen:

1. Ein **Previous Token Head** in einer frühen Schicht hinterlegt bei Token B die Information „vor mir stand A".
2. Der **Induction Head** in einer späteren Schicht sucht beim zweiten A nach einer Stelle, deren Vorgänger A war. Er findet B und kopiert dessen Information.

Im folgenden Beispiel wird ein unsinniger Satz wiederholt. Den zweiten Durchgang kann das Modell nicht aus Weltwissen vorhersagen, nur durch Kopieren. Achten Sie auf einen **diagonalen Streifen in der zweiten Hälfte**, der um die Länge des Satzes versetzt ist: Das zweite „purple" schaut auf das erste „elephant", das zweite „seven" auf das erste „rusty" und so weiter.

In [ ]:
prompt = "The purple elephant juggled seven rusty spoons on the roof. The purple elephant juggled seven rusty spoons on the roof."
chat_modell.zeige_attention(prompt, schicht = [-1])

**Warum sind Induction Heads so wichtig?**

Ein Sprachmodell kennt aus dem Training nur, was in den Trainingsdaten stand. Trotzdem kann es mit Namen, Begriffen und Mustern umgehen, die es nur im aktuellen Text gesehen hat. Man nennt das **In-Context Learning**: Lernen aus dem Kontext, ohne dass sich das Modell selbst verändert. Induction Heads sind ein zentraler Mechanismus dafür:

- **Neue Namen und Begriffe:** Kommt in einem Text „Mr. Thornbury" vor, kann das Modell den Namen später korrekt vervollständigen, obwohl es diesen Menschen nie gesehen hat.
- **Muster aus Beispielen übernehmen:** Gibt man dem Modell im Prompt Beispiele vor (z.B. „Hund → dog, Katze → cat, Maus → ..."), hilft das Kopieren der Struktur bei der Fortsetzung.
- **Programmcode:** Variablennamen, die weiter oben definiert wurden, können später wieder aufgegriffen werden.

<div class="alert alert-secondary">
<details>
<summary>In-Kontext Lernen</summary>

Forschende bei Anthropic haben 2022 beobachtet, dass Induction Heads während des Trainings **recht plötzlich** entstehen. Genau zu diesem Zeitpunkt springt auch die Fähigkeit des Modells zum In-Context Learning sprunghaft nach oben. Das ist ein starkes Indiz dafür, dass beides zusammenhängt.
</div>

<div class="alert alert-secondary">
<details>
<summary>Wiederholungsschleifen</summary>

Die Kehrseite: Dieselbe Kopierneigung trägt vermutlich dazu bei, dass kleine Modelle manchmal in **Wiederholungsschleifen** geraten. Hat ein Modell einen Satz einmal geschrieben, macht der Induction-Mechanismus es wahrscheinlicher, ihn erneut zu schreiben.
</div>

Im nächsten Beispiel sehen Sie den realistischen Anwendungsfall: Der seltene Name „Thornbury" wird in mehrere Tokens zerlegt. Beim zweiten „Thorn" schaut ein Induction Head auf das Token, das beim ersten Mal folgte, und kann so den Namen vervollständigen.

In [ ]:
prompt = "Mr. Thornbury opened the door. Everyone looked at Mr. Thorn"
chat_modell.zeige_attention(prompt, schicht = [-1])